In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from joblib import Parallel, delayed

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)

MOM=0.998
def one_pf(md,z,gr,tw_tvt,tw_gr,ls,ir,gs,N,seed,VN,PN,RESAMP):
    rng=np.random.default_rng(seed)
    pos=ls+2.0*rng.standard_normal(N); rate=ir+0.01*rng.standard_normal(N); w=np.ones(N)/N
    lo,hi=tw_tvt[0]-60,tw_tvt[-1]+60; out=np.empty(len(md)); pm=md[0]-1
    for i in range(len(md)):
        dm=max(md[i]-pm,1.0)
        rate=MOM*rate+VN*rng.standard_normal(N); pos=pos+rate*dm+PN*rng.standard_normal(N)
        tvt=np.clip(pos-z[i],lo,hi); pos=tvt+z[i]
        if np.isfinite(gr[i]):
            d=(gr[i]-np.interp(tvt,tw_tvt,tw_gr))/gs
            lk=np.maximum(np.exp(-0.5*np.minimum(d*d,600)),1e-300)
            w*=lk; s=w.sum(); w=w/s if s>0 else np.ones(N)/N
        if 1.0/(w*w).sum()<RESAMP*N:
            cum=np.cumsum(w); u0=rng.uniform(0,1.0/N)
            idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
            pos=pos[idx]+0.1*rng.standard_normal(N); rate=rate[idx]+0.001*rng.standard_normal(N); w=np.ones(N)/N
        out[i]=float(np.dot(w,pos-z[i])); pm=md[i]
    return out

def analyze(w, NS=40, N=250):
    h=pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv",usecols=["MD","Z","GR","TVT","TVT_input"])
    tw=pd.read_csv(f"{DATA}/train/{w}__typewell.csv").dropna(subset=["TVT","GR"]).sort_values("TVT")
    ps=int(h["TVT_input"].notna().sum())
    if ps<60 or ps>=len(h)-5 or len(tw)<10: return None
    tw_tvt=tw["TVT"].to_numpy(float); tw_gr=tw["GR"].to_numpy(float)
    kn=h.iloc[:ps]; ev=h.iloc[ps:]; tps=float(h["TVT"].iloc[ps-1])
    at=np.interp(kn["TVT_input"].to_numpy(),tw_tvt,tw_gr)
    gs=float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy()-at),10.,60.))
    tail=kn.tail(30); dt=np.diff(tail["TVT_input"].to_numpy()); dz=np.diff(tail["Z"].to_numpy()); dmv=np.diff(tail["MD"].to_numpy()); m=dmv>0
    ir=float(np.median((dt+dz)[m]/dmv[m])) if m.sum()>=3 else 0.
    gr=h["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy()
    idx=ev.index.to_numpy()
    md=np.concatenate([[float(kn["MD"].iloc[-1])],ev["MD"].to_numpy(float)])
    z=np.concatenate([[float(kn["Z"].iloc[-1])],ev["Z"].to_numpy(float)])
    grv=np.concatenate([[np.nan],gr[idx]])
    ls=tps+float(kn["Z"].iloc[-1])
    preds=np.empty((NS,len(md)))
    for s in range(NS):
        pr=np.random.default_rng(1000+s)
        preds[s]=one_pf(md,z,grv,tw_tvt,tw_gr,ls,ir,gs,N,s,0.002*pr.uniform(.9,1.1),0.005*pr.uniform(.85,1.15),pr.uniform(.47,.55))
    P=preds[:,1:]                       # per-seed eval predictions
    y=h["TVT"].to_numpy(float)[idx]
    mean_pred=P.mean(0)
    # bimodality on tail-mean per seed
    tail_mean=P[:,-500:].mean(1) if P.shape[1]>=500 else P.mean(1)
    med=np.median(tail_mean); lowm=tail_mean<med
    sep=abs(tail_mean[~lowm].mean()-tail_mean[lowm].mean()) if lowm.any() and (~lowm).any() else 0.0
    # two mode predictions
    predA=P[lowm].mean(0) if lowm.any() else mean_pred
    predB=P[~lowm].mean(0) if (~lowm).any() else mean_pred
    rA=np.sqrt(np.mean((y-predA)**2)); rB=np.sqrt(np.mean((y-predB)**2))
    oracle=predA if rA<=rB else predB
    return dict(y=y, mean=mean_pred, oracle=oracle, sep=sep, n=len(y))

wells=[os.path.basename(f).split("__")[0] for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))]
print("wells",len(wells),flush=True)
t0=time.time()
res=[r for r in Parallel(n_jobs=4,verbose=10)(delayed(analyze)(w) for w in wells) if r is not None]
def pooled(key):
    e=[(r["y"]-r[key])**2 for r in res]; return float(np.sqrt(np.mean(np.concatenate(e))))
print("used",len(res),"%.0fs"%(time.time()-t0),flush=True)
print("PF mean   pooled %.3f"%pooled("mean"),flush=True)
print("PF oracle-mode pooled %.3f"%pooled("oracle"),flush=True)
seps=np.array([r["sep"] for r in res])
for thr in [10,20,30]:
    bm=[r for r in res if r["sep"]>thr]
    if bm:
        em=np.concatenate([(r["y"]-r["mean"])**2 for r in bm]); eo=np.concatenate([(r["y"]-r["oracle"])**2 for r in bm])
        print(f"bimodal>{thr}ft: {len(bm)} wells  mean {np.sqrt(em.mean()):.2f} -> oracle {np.sqrt(eo.mean()):.2f}",flush=True)
print("DONE",flush=True)